<a href="https://colab.research.google.com/github/thienandinh29/nghiemcuukhoahocsgu2026/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Feature Engineering

**Goal.** Turn raw OHLCV data into a leakage-free feature matrix, add the 1-day and 5-day targets, and create the chronological train / test split used by the model notebooks.

| Item | Value |
|---|---|
| Input | `sp500_supervised_data.csv` (from notebook 01) |
| Features | Returns, trend, momentum, volatility, volume, calendar |
| Targets | `Target_1D`, `Target_5D` — price higher after 1 / 5 trading days |
| Split | Chronological 80 % train / 20 % test (no shuffling) |
| Outputs | `sp500_supervised_features_final.csv`, `sp500_train_80.csv`, `sp500_test_20.csv` |

**Leakage rule.** Every feature at day *t* may only use data up to and including day *t*. This is verified programmatically in section 4.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# ── Shared figure style ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

COLORS = {
    "navy":  "#1F3A5F",   # primary / price
    "teal":  "#2A9D8F",   # positive / strategy
    "amber": "#E9A23B",   # secondary series
    "red":   "#C8553D",   # negative / drawdown
    "grey":  "#6B7280",   # benchmark / text
}
SERIES = [COLORS["teal"], COLORS["amber"], COLORS["red"], COLORS["navy"]]

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.labelcolor": COLORS["grey"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#D1D5DB", "axes.prop_cycle": plt.cycler(color=SERIES),
    "axes.grid": True, "grid.color": "#E5E7EB", "grid.linewidth": 0.8,
    "xtick.color": COLORS["grey"], "ytick.color": COLORS["grey"],
    "legend.frameon": False, "legend.fontsize": 9,
})

## 2. Load raw data

In [ ]:
raw = pd.read_csv("sp500_supervised_data.csv", index_col=0, parse_dates=True).sort_index()
raw.index.name = "Date"

# "Tomorrow" is the future close — it must never become a feature
raw = raw.drop(columns=["Tomorrow", "Target"])

# Zero volume is a data gap, not a real observation: carry the last value forward
raw["Volume"] = raw["Volume"].replace(0, np.nan).ffill()

print(f"{len(raw):,} trading days · {raw.index.min():%Y-%m-%d} → {raw.index.max():%Y-%m-%d}")
raw.head()

## 3. Feature construction

| Group | Features |
|---|---|
| Returns | 1 / 5 / 10 / 20-day return, 1-day log return, lags 1–3 |
| Trend | Close ÷ SMA(5, 10, 20, 50, 200) − 1, SMA50 vs SMA200 spread, MACD (normalised) |
| Momentum | RSI(14), 10-day rate of change |
| Volatility | 10 / 20-day rolling std of returns, ATR(14) ÷ Close, Bollinger %B and width |
| Intraday | High–Low range, Close–Open body, overnight gap (all ÷ Close) |
| Volume | Volume ÷ SMA(20), 1-day volume change |
| Calendar | Day of week |

`Close` is also kept because the back-test in notebook 05 needs it to compute realised returns.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Causal features: row t uses only rows <= t."""
    c, h, l, o, v = df["Close"], df["High"], df["Low"], df["Open"], df["Volume"]
    ret = c.pct_change()
    f = pd.DataFrame(index=df.index)
    f["Close"] = c

    # Returns
    for n in (1, 5, 10, 20):
        f[f"Return_{n}D"] = c.pct_change(n)
    f["LogReturn_1D"] = np.log(c / c.shift(1))
    for k in (1, 2, 3):
        f[f"Return_Lag{k}"] = ret.shift(k)

    # Trend
    for n in (5, 10, 20, 50, 200):
        f[f"Close_vs_SMA{n}"] = c / c.rolling(n).mean() - 1
    f["SMA50_vs_SMA200"] = c.rolling(50).mean() / c.rolling(200).mean() - 1
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    f["MACD"] = macd / c
    f["MACD_Signal"] = macd.ewm(span=9, adjust=False).mean() / c

    # Momentum — RSI with Wilder smoothing
    delta = c.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    f["RSI_14"] = 100 - 100 / (1 + gain / loss)
    f["ROC_10"] = c.pct_change(10)

    # Volatility
    f["Volatility_10D"] = ret.rolling(10).std()
    f["Volatility_20D"] = ret.rolling(20).std()
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    f["ATR_14_pct"] = tr.ewm(alpha=1 / 14, adjust=False).mean() / c
    sma20, std20 = c.rolling(20).mean(), c.rolling(20).std()
    upper, lower = sma20 + 2 * std20, sma20 - 2 * std20
    f["Bollinger_PctB"] = (c - lower) / (upper - lower)
    f["Bollinger_Width"] = (upper - lower) / sma20

    # Intraday structure
    f["HL_Range"] = (h - l) / c
    f["Body"] = (c - o) / c
    f["Gap"] = o / c.shift() - 1

    # Volume
    f["Volume_vs_SMA20"] = v / v.rolling(20).mean()
    f["Volume_Change_1D"] = v.pct_change()

    # Calendar
    f["DayOfWeek"] = df.index.dayofweek
    return f

features = build_features(raw)
print(f"{features.shape[1]} features")
features.describe().T[["mean", "std", "min", "max"]].round(4)

### Targets

Both targets compare a *future* close with today's close, so the last 1 / 5 rows have no label and are dropped together with the indicator warm-up period (200 days for SMA200).

In [ ]:
close = raw["Close"]
features["Target_1D"] = (close.shift(-1) > close).astype(int)
features["Target_5D"] = (close.shift(-5) > close).astype(int)

# Rows with an unlabelled future (last 5 days) or incomplete indicators (first ~200 days)
unlabeled = close.shift(-5).isna()
dataset = features[~unlabeled].dropna()
assert np.isfinite(dataset.select_dtypes("number")).all().all(), "inf/NaN left in dataset"

print(f"Dropped {len(features) - len(dataset):,} rows (indicator warm-up + unlabelled tail)")
print(f"Final dataset: {dataset.shape[0]:,} rows × {dataset.shape[1]} columns")

## 4. Leakage check

Re-compute the features on data truncated at a past date. If every feature is causal, the values on that date must be **identical** to those computed from the full history.

In [ ]:
cutoff = dataset.index[len(dataset) // 2]
truncated = build_features(raw.loc[:cutoff])

feature_cols = [c for c in features.columns if not c.startswith("Target")]
diff = (truncated.loc[cutoff, feature_cols] - features.loc[cutoff, feature_cols]).abs().max()
assert diff < 1e-9, f"Look-ahead detected (max diff {diff:.2e})"
print(f"✓ Causality check passed at {cutoff:%Y-%m-%d} (max diff {diff:.1e})")

## 5. Feature and target diagnostics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.6, 1]})

# A · Correlation of each feature with the 1-day target
corr = dataset.drop(columns=["Target_5D"]).corr()["Target_1D"].drop("Target_1D").sort_values()
ax1.barh(corr.index, corr.values, color=np.where(corr > 0, COLORS["teal"], COLORS["red"]))
ax1.set(title="Correlation with Target_1D", xlabel="Pearson r")
ax1.grid(axis="y", visible=False)

# B · Class balance for both horizons
bal = dataset[["Target_1D", "Target_5D"]].mean() * 100
bars = ax2.bar(["1-day", "5-day"], bal.values, color=[COLORS["navy"], COLORS["amber"]], width=0.5)
ax2.axhline(50, color=COLORS["grey"], linestyle="--", linewidth=1)
for b, v in zip(bars, bal.values):
    ax2.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", color=COLORS["grey"])
ax2.set(title="Share of 'up' days", ylabel="%", ylim=(0, 75))
ax2.grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

> Individual correlations with the target are tiny (|r| ≪ 0.1) — expected for daily index direction. Do not read this as a bug; it sets realistic expectations for the model notebooks.

## 6. Chronological train / test split

The data is split **by time**, never shuffled, so the test set is strictly later than the training set.

In [ ]:
split = int(len(dataset) * 0.8)
train_df, test_df = dataset.iloc[:split], dataset.iloc[split:]

print(f"Train: {len(train_df):,} rows  {train_df.index.min():%Y-%m-%d} → {train_df.index.max():%Y-%m-%d}")
print(f"Test:  {len(test_df):,} rows  {test_df.index.min():%Y-%m-%d} → {test_df.index.max():%Y-%m-%d}")
assert train_df.index.max() < test_df.index.min()

## 7. Export

In [ ]:
dataset.to_csv("sp500_supervised_features_final.csv")
train_df.to_csv("sp500_train_80.csv")
test_df.to_csv("sp500_test_20.csv")
print("Saved: sp500_supervised_features_final.csv, sp500_train_80.csv, sp500_test_20.csv")